### Permutation ANOVA Analysis


In [26]:
# Notebook header
%load_ext autoreload
%autoreload 2
import pandas as pd
from analyses.spike_count import get_binned_spike_trials, aggregate_trial_level
from analyses.statistical_tests import perform_statistical_test_on_dataframe_rows, permutation_anova_test

import warnings
from scipy.stats import ConstantInputWarning

warnings.simplefilter("ignore", ConstantInputWarning)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [30]:
# Step 1: Load and prepare data
date = "2023-09-26"
round_no = 2
bin_size = 0.05

analysis_df = get_binned_spike_trials(date, round_no, bin_size)
filtered_df = analysis_df[analysis_df['MonkeyGroup'] == 'Zombies']
trial_level_df = aggregate_trial_level(filtered_df)


Reading Intan Technologies RHD2000 Data File, Version 3.2

Found 24 amplifier channels.
Found 0 auxiliary input channels.
Found 0 supply voltage channels.
Found 0 board ADC channels.
Found 2 board digital input channels.
Found 0 board digital output channels.
Found 0 temperature sensors channels.

Header file contains no data.  Amplifiers were sampled at 20.00 kS/s.
Done!  Elapsed time: 0.0 seconds


In [31]:
# Step 2: Run permutation ANOVA (trial-level, by MonkeyName)
all_results = []
unique_neurons = trial_level_df['NeuronID'].unique()
for neuron_id in unique_neurons:
    neuron_df = trial_level_df[trial_level_df['NeuronID'] == neuron_id]
    grouped = neuron_df.groupby('MonkeyName')['SpikeCount'].apply(list)
    if len(grouped) < 2:
        continue
    anova_input_df = pd.DataFrame([grouped])
    results, _ = perform_statistical_test_on_dataframe_rows(
        anova_input_df,
        test_func=permutation_anova_test,
        num_permutations=1000
    )
    for result in results:
        index, f_stat, p_value = result
        all_results.append({
            'NeuronID': neuron_id,
            'F-statistic': f_stat,
            'p-value': p_value
        })


Row SpikeCount: Skipped — NaN result.


In [32]:
# Step 3: Process and display results
results_df = pd.DataFrame(all_results)
significant_df = results_df[results_df['p-value'] < 0.05]
print(significant_df)

Empty DataFrame
Columns: [NeuronID, F-statistic, p-value]
Index: []
